# 基于 ELECTRA-DPCNN 的餐饮评论细粒度情感分析

本笔记本对应论文 *Fine Grained Sentiment Analysis of Catering Reviews Based on ELECTRA-DPCNN Model*，任务为**方面级情感分析（ABSA）**：给定一条餐饮评论和一个方面类别，预测该方面上的情感极性（正面 / 中性 / 负面）。

模型结构：ELECTRA-small 编码 `[CLS] 评论 [SEP] 方面 [SEP]`，再经改进 DPCNN（Region Embedding → 窄卷积 → 残差卷积块 → 下采样 → 自注意力）做多尺度特征抽象，最后三分类。

默认读取 `data/ASAP_ASPECT/` 下的官方划分；若该目录不存在且根目录有 `ASAP_ASPECT.zip`，会自动解压。训练权重与曲线保存在 `outputs/`。


## 1. 环境与数据准备

依赖见 `requirements.txt`。在百度 AI Studio 上运行时，若数据已挂载到 `/home/aistudio/data/data347891`，会自动使用该路径。


In [ ]:
import math
import os
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import paddle
import paddle.nn as nn
import paddle.nn.functional as F
from paddle.io import DataLoader, Dataset
from paddlenlp.transformers import ElectraModel, ElectraTokenizer

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def resolve_data_dir():
    zip_path = ROOT / "ASAP_ASPECT.zip"
    local_dir = ROOT / "data" / "ASAP_ASPECT"
    if not _has_split_files(local_dir) and zip_path.exists():
        print(f"解压数据集: {zip_path} -> {ROOT / 'data'}")
        with zipfile.ZipFile(zip_path, "r") as zf:
            for member in zf.namelist():
                if member.endswith("/") or ".DS_Store" in member or member.startswith("__MACOSX"):
                    continue
                zf.extract(member, ROOT / "data")
    candidates = [
        local_dir,
        Path("/home/aistudio/data/data347891"),
        Path("/home/aistudio/data"),
    ]
    for path in candidates:
        if _has_split_files(path):
            return path
    raise FileNotFoundError(
        "未找到 ASAP_ASPECT 数据。请将 ASAP_ASPECT.zip 放在项目根目录，"
        "或把 train.tsv / trian.tsv、dev.tsv、test.tsv 放到 data/ASAP_ASPECT/"
    )


def _has_split_files(path):
    path = Path(path)
    if not path.is_dir():
        return False
    has_train = (path / "train.tsv").exists() or (path / "trian.tsv").exists()
    return has_train and (path / "dev.tsv").exists() and (path / "test.tsv").exists()


def resolve_train_path(data_dir):
    data_dir = Path(data_dir)
    for name in ("train.tsv", "trian.tsv"):
        candidate = data_dir / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"在 {data_dir} 中未找到 train.tsv 或 trian.tsv")


DATA_DIR = resolve_data_dir()
print(f"数据目录: {DATA_DIR}")
print(f"训练文件: {resolve_train_path(DATA_DIR).name}")


## 2. 配置

超参与论文实验设置一致。`use_official_dev=True` 时使用官方 `dev.tsv` 做验证；设为 `False` 时会从训练集按 8:2 随机切分（与原 AI Studio 脚本行为一致）。


In [ ]:
class Config:
    data_dir = DATA_DIR
    train_path = str(resolve_train_path(DATA_DIR))
    dev_path = str(DATA_DIR / "dev.tsv")
    test_path = str(DATA_DIR / "test.tsv")
    result_path = str(OUTPUT_DIR / "ASAP_ASPECT.tsv")
    model_path = str(OUTPUT_DIR / "best_model.pdparams")
    pretrained_model = "electra-small"
    max_seq_len = 128
    batch_size = 32
    num_epochs = 50
    learning_rate = 2e-5
    num_classes = 3
    conv_filters = 256
    kernel_size = 3
    dropout_prob = 0.1
    l2_lambda = 1e-4
    hidden_size = 256
    patience = 15
    use_official_dev = True
    train_split = 0.8
    seed = 42
    num_workers = 0


config = Config()
paddle.seed(config.seed)
np.random.seed(config.seed)
device = paddle.set_device("gpu" if paddle.is_compiled_with_cuda() else "cpu")
print(f"device: {device}")
print(
    f"train={config.train_path}\n"
    f"dev={config.dev_path}\n"
    f"test={config.test_path}\n"
    f"use_official_dev={config.use_official_dev}"
)


## 3. 数据集

| 划分 | 文件 | 样本数 | 表头 |
|------|------|--------|------|
| 训练 | `train.tsv`（原始文件名为 `trian.tsv`） | 213,371 | `text_a`, `cate`, `label` |
| 验证 | `dev.tsv` | 29,101 | `qid`, `text_a`, `cate`, `label` |
| 测试 | `test.tsv` | 28,362 | `qid`, `text_a`, `cate` |

标签为 `{1, 0, -1}`，读入时映射到 `{2, 1, 0}` 以便交叉熵训练；预测阶段再映射回原标签。输入编码为 `[CLS] 评论文本 [SEP] 方面类别 [SEP]`。


In [ ]:
def read_tsv(path):
    try:
        return pd.read_csv(path, sep="\t", header=0, on_bad_lines="skip")
    except TypeError:
        return pd.read_csv(path, sep="\t", header=0, error_bad_lines=False)


class AspectDataset(Dataset):
    """方面级情感分析数据集：评论文本 + 方面类别 -> 极性标签。"""

    def __init__(self, data_path, tokenizer, is_test=False):
        self.tokenizer = tokenizer
        self.is_test = is_test
        self.data = self._load_data(data_path)

    def _load_data(self, path):
        df = read_tsv(path)
        required = ["text_a", "cate"] if self.is_test else ["text_a", "cate", "label"]
        missing = [col for col in required if col not in df.columns]
        if missing:
            raise ValueError(f"{path} 缺少列 {missing}，实际列: {list(df.columns)}")

        texts = df["text_a"].fillna("").astype(str).tolist()
        aspects = df["cate"].fillna("").astype(str).tolist()

        if self.is_test:
            if "qid" in df.columns:
                qids = pd.to_numeric(df["qid"], errors="coerce").fillna(0).astype(int).tolist()
            else:
                qids = list(range(len(df)))
            return [
                {"qid": qid, "text": text, "aspect": aspect}
                for qid, text, aspect in zip(qids, texts, aspects)
            ]

        labels = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int) + 1
        return [
            {"text": text, "aspect": aspect, "label": int(label)}
            for text, aspect, label in zip(texts, aspects, labels.tolist())
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        encoded = self.tokenizer(
            text=item["text"],
            text_pair=item["aspect"],
            max_seq_len=config.max_seq_len,
            pad_to_max_seq_len=True,
            return_attention_mask=True,
            return_token_type_ids=True,
        )
        sample = {
            "input_ids": np.array(encoded["input_ids"], dtype="int64"),
            "token_type_ids": np.array(encoded["token_type_ids"], dtype="int64"),
            "attention_mask": np.array(encoded["attention_mask"], dtype="int64"),
        }
        if self.is_test:
            sample["qid"] = np.array(item["qid"], dtype="int64")
        else:
            sample["labels"] = np.array(item["label"], dtype="int64")
        return sample


## 4. 模型

### 4.1 改进 DPCNN

在经典 DPCNN 上增加两处核大小为 1 的窄卷积（分别接在 Region Embedding 与下采样之后），并在金字塔结构末端加入自注意力，用于强化长评论中的情感线索、抑制噪声。


In [ ]:
class DPCNN(nn.Layer):
    """Region Embedding + 窄卷积 + 残差金字塔下采样 + 自注意力。"""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1D(
            in_channels=config.hidden_size,
            out_channels=config.conv_filters,
            kernel_size=config.kernel_size,
            padding=config.kernel_size // 2,
        )
        self.narrow_conv_after_embedding = nn.Conv1D(
            in_channels=config.conv_filters,
            out_channels=config.conv_filters,
            kernel_size=1,
        )
        self.conv_blocks = nn.LayerList(
            [
                nn.Conv1D(
                    in_channels=config.conv_filters,
                    out_channels=config.conv_filters,
                    kernel_size=config.kernel_size,
                    padding=config.kernel_size // 2,
                )
                for _ in range(2)
            ]
        )
        self.downsample = nn.MaxPool1D(kernel_size=3, stride=2, padding=1)
        self.narrow_conv_after_pooling = nn.Conv1D(
            in_channels=config.conv_filters,
            out_channels=config.conv_filters,
            kernel_size=1,
        )
        self.shortcut = nn.Conv1D(
            in_channels=config.conv_filters,
            out_channels=config.conv_filters,
            kernel_size=1,
        )
        self.attn_query = nn.Linear(config.conv_filters, config.conv_filters)
        self.attn_key = nn.Linear(config.conv_filters, config.conv_filters)
        self.attn_value = nn.Linear(config.conv_filters, config.conv_filters)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.narrow_conv_after_embedding(x))
        residual = x

        for conv in self.conv_blocks:
            out = F.relu(conv(x))
            out = conv(out)
            if out.shape[2] == residual.shape[2]:
                shortcut = (
                    self.shortcut(residual)
                    if residual.shape[1] != config.conv_filters
                    else residual
                )
                out = F.relu(out + shortcut)
            out = self.downsample(out)
            out = F.relu(self.narrow_conv_after_pooling(out))
            residual = self.downsample(residual)
            x = out

        x = x.transpose([0, 2, 1])
        query = self.attn_query(x)
        key = self.attn_key(x)
        value = self.attn_value(x)
        attn_scores = paddle.matmul(query, key, transpose_y=True) / math.sqrt(config.conv_filters)
        attn_weights = F.softmax(attn_scores, axis=-1)
        attn_output = paddle.matmul(attn_weights, value)
        return attn_output.transpose([0, 2, 1])


### 4.2 ELECTRA-DPCNN

ELECTRA 最后一层隐状态转置为 `[batch, hidden, seq]` 后送入 DPCNN，自适应平均池化得到句向量，再经 Dropout 与线性层输出三类 logits。


In [ ]:
class ElectraDPCNN(nn.Layer):
    def __init__(self):
        super().__init__()
        self.electra = ElectraModel.from_pretrained(config.pretrained_model)
        self.dpcnn = DPCNN()
        self.dropout = nn.Dropout(config.dropout_prob)
        self.classifier = nn.Linear(config.conv_filters, config.num_classes)

    def forward(self, input_ids, token_type_ids, attention_mask):
        electra_output = self.electra(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask,
        )
        hidden_states = electra_output[0] if isinstance(electra_output, tuple) else electra_output
        conv_input = hidden_states.transpose([0, 2, 1])
        dpcnn_output = self.dpcnn(conv_input)
        pooled = F.adaptive_avg_pool1d(dpcnn_output, 1).squeeze(2)
        logits = self.classifier(self.dropout(pooled))
        return logits


## 5. 训练

优化器为 AdamW（学习率 `2e-5`，权重衰减 `1e-4`），损失为交叉熵。按验证集准确率保存最优权重，连续 15 个 epoch 无提升则早停。训练结束后写出损失 / 准确率曲线。


In [ ]:
def build_train_dev_datasets(tokenizer):
    if config.use_official_dev:
        train_dataset = AspectDataset(config.train_path, tokenizer)
        dev_dataset = AspectDataset(config.dev_path, tokenizer)
        print(f"使用官方划分  train={len(train_dataset)}  dev={len(dev_dataset)}")
        return train_dataset, dev_dataset

    full_dataset = AspectDataset(config.train_path, tokenizer)
    train_size = int(config.train_split * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, dev_dataset = paddle.io.random_split(full_dataset, [train_size, val_size])
    print(f"从训练集随机切分  train={train_size}  dev={val_size}")
    return train_dataset, dev_dataset


def plot_curves(epoch_losses, epoch_accuracies, save_dir):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    epochs_range = range(1, len(epoch_losses) + 1)

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_range, epoch_losses, marker="o", linestyle="-", color="b")
    plt.title("Training Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Average Loss")
    plt.grid(True)
    plt.xticks(list(epochs_range))
    plt.tight_layout()
    plt.savefig(save_dir / "training_loss_curve.png")
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_range, epoch_accuracies, marker="o", linestyle="-", color="r")
    plt.title("Validation Accuracy Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1)
    plt.grid(True)
    plt.xticks(list(epochs_range))
    plt.tight_layout()
    plt.savefig(save_dir / "validation_accuracy_curve.png")
    plt.close()

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, epoch_losses, marker="o", linestyle="-", color="b")
    plt.title("Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.xticks(list(epochs_range))
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, epoch_accuracies, marker="o", linestyle="-", color="r")
    plt.title("Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1)
    plt.grid(True)
    plt.xticks(list(epochs_range))
    plt.tight_layout()
    plt.savefig(save_dir / "combined_metrics.png")
    plt.close()
    print(f"曲线已保存到 {save_dir}")


def evaluate(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with paddle.no_grad():
        for batch in data_loader:
            logits = model(batch["input_ids"], batch["token_type_ids"], batch["attention_mask"])
            preds = paddle.argmax(logits, axis=1)
            correct += paddle.sum(preds == batch["labels"]).numpy().item()
            total += len(batch["labels"])
    return 0.0 if total == 0 else correct / total


def train():
    tokenizer = ElectraTokenizer.from_pretrained(config.pretrained_model)
    model = ElectraDPCNN()
    model.to(device)

    train_dataset, dev_dataset = build_train_dev_datasets(tokenizer)
    train_loader = DataLoader(
        train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=config.num_workers
    )
    dev_loader = DataLoader(
        dev_dataset, batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers
    )

    optimizer = paddle.optimizer.AdamW(
        learning_rate=config.learning_rate,
        parameters=model.parameters(),
        weight_decay=config.l2_lambda,
    )
    criterion = nn.CrossEntropyLoss()

    best_acc = 0.0
    epochs_no_improve = 0
    epoch_losses = []
    epoch_accuracies = []

    for epoch in range(config.num_epochs):
        model.train()
        total_loss = 0.0
        for step, batch in enumerate(train_loader):
            logits = model(batch["input_ids"], batch["token_type_ids"], batch["attention_mask"])
            loss = criterion(logits, batch["labels"])
            loss.backward()
            optimizer.step()
            optimizer.clear_grad()
            total_loss += loss.numpy().item()
            if step % 50 == 0:
                print(f"Epoch {epoch + 1}, Step {step}, Loss: {loss.numpy().item():.4f}")

        acc = evaluate(model, dev_loader)
        avg_loss = 0.0 if len(train_loader) == 0 else total_loss / len(train_loader)
        epoch_losses.append(avg_loss)
        epoch_accuracies.append(acc)
        print(
            f"Epoch {epoch + 1}/{config.num_epochs}, "
            f"Avg Loss: {avg_loss:.4f}, Dev Acc: {acc:.4f}"
        )

        if acc > best_acc:
            best_acc = acc
            epochs_no_improve = 0
            paddle.save(model.state_dict(), config.model_path)
            print(f"Saved best model with accuracy: {best_acc:.4f}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= config.patience:
                print(f"Early stopping triggered after {config.patience} epochs with no improvement.")
                break

    print(f"Training completed. Best Dev Acc: {best_acc:.4f}")
    plot_curves(epoch_losses, epoch_accuracies, OUTPUT_DIR)
    return best_acc


## 6. 预测

加载验证集上最优权重，对测试集推理，将类别索引减 1 还原为 `{-1, 0, 1}`，结果写入 `outputs/ASAP_ASPECT.tsv`。


In [ ]:
def predict():
    if not os.path.exists(config.model_path):
        print(f"未找到权重文件 {config.model_path}，请先训练。")
        return

    tokenizer = ElectraTokenizer.from_pretrained(config.pretrained_model)
    model = ElectraDPCNN()
    model.set_state_dict(paddle.load(config.model_path))
    model.to(device)
    model.eval()

    test_dataset = AspectDataset(config.test_path, tokenizer, is_test=True)
    test_loader = DataLoader(
        test_dataset, batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers
    )

    predictions = []
    qids = []
    with paddle.no_grad():
        for batch in test_loader:
            logits = model(batch["input_ids"], batch["token_type_ids"], batch["attention_mask"])
            preds = paddle.argmax(logits, axis=1).numpy() - 1
            predictions.extend(preds.tolist())
            qids.extend(batch["qid"].numpy().reshape(-1).tolist())

    result_df = pd.DataFrame({"qid": qids, "prediction": predictions})
    result_df.to_csv(config.result_path, sep="\t", index=False)
    print(f"预测结果已保存: {config.result_path}  ({len(result_df)} 条)")
    return result_df


## 7. 运行

将下面两个开关设为 `True` 后运行该单元格。完整 50 epoch 训练在 Tesla V100 上耗时较长，可先把 `config.num_epochs` 调小做通路检查。


In [ ]:
DO_TRAIN = True
DO_PREDICT = True

if DO_TRAIN:
    train()
if DO_PREDICT:
    predict()
